# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarim-Ali-Shah/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### The Data Contract in Plain Words (5 Core Answers)

1. **What one row means (Unit of Analysis):**
   - At the warehouse daily level: **One daily observation of one content item for one client** (Grain: `report_date x client_hash_id x content_hash_id`).
   - At the model feature level: **One unique content page (`content_hash_id`) for a client (`client_hash_id`) aggregated over the chosen feature window.**

2. **Which table(s) we will use:**
   - Primary Fact Table: `fact_content_daily_performance` (partitioned by month; iterating on `month=2026-03`).
   - Dimension Tables: `dim_content` (page metadata and content taxonomy) and `dim_clients` (client access profiles and tracking start dates).

3. **Which time window:**
   - Mid-panel development partition: **March 2026 (`2026-03-01` to `2026-03-31`, 31 days).**
   - Feature Window: Early month (`2026-03-01` to `2026-03-20`, 20 days).
   - Target Outcome Window: Late month (`2026-03-21` to `2026-03-31`, 11 days).

4. **What we predict or rank (Label / Proxy):**
   - Target: `is_declining_target` — an observed forward-window drop in daily impression velocity ($\ge 30\%$ drop in late-month daily impressions compared to the early-month baseline rate).

5. **One thing we deliberately EXCLUDE:**
   - We strictly exclude any metrics calculated from the forward window from entering the feature set, and we exclude rows where `ga4_data_available = FALSE` when calculating GA4 user engagement rates.


In [1]:
import os, sys, getpass
import duckdb
import pandas as pd
import numpy as np

# 1. Authenticate DuckDB against Hugging Face warehouse
# Secure token resolution: env var -> Colab Secret -> prompt
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
SRC_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("DuckDB initialized with hf:// connector.")
print("Target development slice: fact_content_daily_performance (month=2026-03)")


DuckDB initialized with hf:// connector.
Target development slice: fact_content_daily_performance (month=2026-03)


## 2. Fields: feature / label / context / excluded

Every column used in our analysis falls strictly into one of four buckets:

| Field | Bucket | Why it belongs here |
|---|---|---|
| `gsc_impressions` (early window) | **Feature** | Observable Search Console exposure prior to prediction cutoff. |
| `gsc_clicks` (early window) | **Feature** | Observable Search Console traffic prior to prediction cutoff. |
| `gsc_avg_position` (early window) | **Feature** | Observable ranking tier before prediction cutoff. |
| `ga4_engaged_sessions` (early window) | **Feature** | Observable on-page engagement; filtered with `ga4_data_available IS TRUE`. |
| `scroll_events` (early window) | **Feature** | Observable user interaction depth. |
| `is_declining_target` | **Label / Proxy** | Observed outcome in the subsequent forward window (never an input). |
| `client_hash_id` | **Context** | Grouping and grouped client-holdout validation splits only. |
| `content_hash_id` | **Context** | Unique join key connecting daily facts to `dim_content`. |
| `leaky_forward_clicks` | **Excluded** | Future outcome information that occurs after decision cutoff (causes 100% data leakage). |
| Unfiltered GA4 rows | **Excluded** | Rows before client onboarding where GA4 is zero-filled with `ga4_data_available = FALSE`. |


In [2]:
# Verify column presence in dim_content and fact_daily
cols_daily = con.sql(f"DESCRIBE SELECT * FROM {SRC_MARCH} LIMIT 1").df()
print(f"Total columns available in fact_content_daily_performance: {len(cols_daily)}")

# Print selected key fields across buckets
selected_fields = ["report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_engaged_sessions", "ga4_data_available"]
print("Sample metadata verified for key fields:")
print(cols_daily[cols_daily["column_name"].isin(selected_fields)][["column_name", "column_type"]].to_string(index=False))


Total columns available in fact_content_daily_performance: 31
Sample metadata verified for key fields:
         column_name column_type
         report_date        DATE
      client_hash_id     VARCHAR
     content_hash_id     VARCHAR
  ga4_data_available     BOOLEAN
     gsc_impressions      BIGINT
          gsc_clicks      BIGINT
    gsc_avg_position      DOUBLE
ga4_engaged_sessions      BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

### Proving 3 Facts on `month=2026-03`:
1. **Query 1 — The Grain:** Proves that `report_date x client_hash_id x content_hash_id` has zero duplicate rows.
2. **Query 2 — Slice Size & Span:** Proves row counts, date bounds (`2026-03-01` to `2026-03-31`), and distinct entity counts.
3. **Query 3 — Availability Filter (`IS TRUE`):** Proves how many rows have verified Search Console and Analytics tracking active.

### Building the 5-Feature Frame & The Leakage Trap:
- **Feature 1 (`f1_impressions_early`):** Total GSC impressions in days 1–20. *Knowable at decision moment because GSC syncs daily.*
- **Feature 2 (`f2_clicks_early`):** Total GSC clicks in days 1–20. *Knowable at decision moment from GSC.*
- **Feature 3 (`f3_avg_position_early`):** Average rank position in days 1–20. *Knowable at decision moment.*
- **Feature 4 (`f4_engaged_sessions_early`):** Total GA4 engaged sessions in days 1–20. *Knowable at decision moment when `ga4_data_available IS TRUE`.*
- **Feature 5 (`f5_scroll_events_early`):** Total scroll events in days 1–20. *Knowable at decision moment.*
- **The Trap:** We inject `leaky_forward_clicks` (clicks occurring in days 21–31) to observe an artificial leap in ROC-AUC, then remove it to enforce honest evaluation.


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# --- Query 1: Verify Daily Grain ---
q1_grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as cnt
    FROM {SRC_MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING cnt > 1
    LIMIT 5
""").df()
print("Query 1 (Grain Check - Duplicates at daily grain):", len(q1_grain), "rows (0 duplicates = grain holds!)")

# --- Query 2: Slice Row Count and Date Span ---
q2_counts = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items
    FROM {SRC_MARCH}
""").df()
print("\nQuery 2 (Slice Size and Bounds):")
print(q2_counts.to_string(index=False))

# --- Query 3: Availability Filter with IS TRUE ---
q3_avail = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS gsc_available_pct,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS ga4_available_pct
    FROM {SRC_MARCH}
""").df()
print("\nQuery 3 (Availability Check):")
print(q3_avail.to_string(index=False))

# --- Build 5-Feature Frame + The Trap ---
feature_df = con.sql(f"""
    WITH history AS (
        SELECT 
            client_hash_id,
            content_hash_id,
            -- 5 safe features knowable at cutoff (March 20, 2026)
            COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_impressions ELSE 0 END), 0) AS f1_impressions_early,
            COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_clicks ELSE 0 END), 0) AS f2_clicks_early,
            COALESCE(AVG(CASE WHEN report_date <= DATE '2026-03-20' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END), 0.0) AS f3_avg_position_early,
            COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-20' AND ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END), 0) AS f4_engaged_sessions_early,
            COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN scroll_events ELSE 0 END), 0) AS f5_scroll_events_early,
            
            -- Future outcome (March 21 - March 31)
            COALESCE(SUM(CASE WHEN report_date > DATE '2026-03-20' THEN gsc_impressions ELSE 0 END), 0) AS imp_late,
            -- Leaked column from future window
            COALESCE(SUM(CASE WHEN report_date > DATE '2026-03-20' THEN gsc_clicks ELSE 0 END), 0) AS leaky_forward_clicks
        FROM {SRC_MARCH}
        GROUP BY client_hash_id, content_hash_id
        HAVING SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_impressions ELSE 0 END) >= 50
    )
    SELECT *,
        CASE WHEN (imp_late / 11.0) < (f1_impressions_early / 20.0 * 0.70) THEN 1 ELSE 0 END AS is_declining_target
    FROM history
    LIMIT 10000
""").df()

print(f"\n5-Feature Frame shape: {feature_df.shape}")

# Evaluate Honest vs Leaky Model
honest_features = ["f1_impressions_early", "f2_clicks_early", "f3_avg_position_early", "f4_engaged_sessions_early", "f5_scroll_events_early"]
X_honest = feature_df[honest_features]
y = feature_df["is_declining_target"]

clf_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_auc = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"Honest Model ROC-AUC : {honest_auc:.3f}")

# The Trap: Add leaky future feature
X_leaky = X_honest.copy()
X_leaky["leaky_forward_clicks"] = feature_df["leaky_forward_clicks"]
clf_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, clf_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky Trap ROC-AUC   : {leaky_auc:.3f} (Artificially inflated!)")

# Remove the leaked feature
del X_leaky
print("Leakage resolved: Leaked future column discarded; retained honest features.")


Query 1 (Grain Check - Duplicates at daily grain): 0 rows (0 duplicates = grain holds!)



Query 2 (Slice Size and Bounds):
 total_rows   min_date   max_date  distinct_clients  distinct_content_items
    9841378 2026-03-01 2026-03-31                55                  331437



Query 3 (Availability Check):
 total_rows  gsc_available_rows  gsc_available_pct  ga4_available_rows  ga4_available_pct
    9841378             3611061              36.69              413966               4.21



5-Feature Frame shape: (10000, 10)
Honest Model ROC-AUC : 0.617
Leaky Trap ROC-AUC   : 0.671 (Artificially inflated!)
Leakage resolved: Leaked future column discarded; retained honest features.


## 4. Data limits

### Key Limitations of this Warehouse Slice:
1. **Unbalanced History Across Clients:**
   - Different clients started tracking at vastly different points in time (`gsc_data_start` vs `ga4_data_start`). Only a minority of clients possess 12+ months of continuous history.
2. **GA4 Sparsity & Zero-Fill Danger:**
   - As proven in Query 3, only 4.21% of daily rows in March 2026 have active GA4 tracking (`ga4_data_available = TRUE`).
   - Rows prior to client GA4 activation have GA4 metrics set to 0. A blind query taking `AVG(ga4_engaged_sessions)` would mistake 'no tracking enabled' for 'zero user engagement.' Always filter with `WHERE ga4_data_available IS TRUE`.
3. **Observational Bounds:**
   - The warehouse provides observational search behavior; it cannot reveal competitor content edits, Google search algorithm changes, or off-SERP user context.


In [4]:
# Query dim_clients to show history start date variation across clients
client_history = con.sql(f"""
    SELECT 
        access_profile,
        COUNT(*) as n_clients,
        MIN(gsc_data_start) as earliest_gsc,
        MAX(gsc_data_start) as latest_gsc,
        COUNT(*) FILTER (WHERE ga4_data_start IS NOT NULL) as clients_with_ga4
    FROM read_parquet('{REL}/dim_clients.parquet')
    GROUP BY access_profile
""").df()
print("=== Client History Limits Audit (dim_clients) ===")
print(client_history.to_string(index=False))


=== Client History Limits Audit (dim_clients) ===
                      access_profile  n_clients earliest_gsc latest_gsc  clients_with_ga4
                            ga4_only          1          NaT        NaT                 0
                         gsc_and_ga4         53   2025-01-27 2026-06-02                50
       no_search_or_analytics_access         26   2025-11-05 2025-11-05                 0
                            gsc_only         14   2025-06-07 2026-04-07                 0
source_only_missing_client_dimension         10   2025-11-05 2025-11-07                 1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
